# 📝 과제 02: 온라인쇼핑 조문 검색기를 hard negative로 학습하기

## 만들 결과

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">과제 01에서 저장한 쌍 학습 모델로 온라인쇼핑 학습 질의마다 hard negative를 뽑고 거짓 오답 행을 뺀 triplet으로 새 모델을 학습하고, 베이스 모델·쌍 학습 모델과 비교합니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">만들 것</strong></p>
<ul>
<li>베이스 모델 <code style="font-size:0.95em;color:#345473;">base_model</code>과 과제 01 폴더에서 불러온 쌍 학습 모델 <code style="font-size:0.95em;color:#345473;">commerce_tuned_model</code></li>
<li>학습 질의 하나에서 직접 뽑은 hard negative <code style="font-size:0.95em;color:#345473;">hard_neg_text</code></li>
<li><code style="font-size:0.95em;color:#345473;">mine_hard_negatives</code>로 만든 triplet 데이터셋 <code style="font-size:0.95em;color:#345473;">triplet_ds</code>(베이스 모델로 뽑은 <code style="font-size:0.95em;color:#345473;">base_mined</code>와 행 수 비교)</li>
<li>검토 목록의 거짓 오답 행을 뺀 <code style="font-size:0.95em;color:#345473;">triplet_ds</code>와 뺀 행 테이블 <code style="font-size:0.95em;color:#345473;">removed_table</code></li>
<li>triplet으로 학습한 hard negative 학습 모델 <code style="font-size:0.95em;color:#345473;">commerce_hard_model</code></li>
<li>세 모델을 전체, 학습에 나온 조문 질의, 처음 보는 조문 질의로 나눠 비교한 테이블 <code style="font-size:0.95em;color:#345473;">compare_table</code></li>
<li>일반 83질의의 모델별 정답 순위 <code style="font-size:0.95em;color:#345473;">general_model_ranks</code>와 전체·분야별 세 모델 테이블 <code style="font-size:0.95em;color:#345473;">general_score_table</code></li>
</ul></div>

## 준비: 데이터와 제공 코드

- **먼저 할 일:** 과제 01을 7번까지 실행해 `output/commerce_tuned`에 쌍 학습 모델을 저장해 둡니다.
- **자료:** 과제 01과 같은 `data/과제/commerce_*.jsonl` 세 파일, 마지막 7번에서 읽는 `data/본실습/general_corpus.jsonl`(433문서)과 `general_eval.jsonl`(83질의)
- **제공 코드:** 준비 셀, 교안 01과 같은 함수 네 개(`load_jsonl`, `encode_texts`, `gold_ranks`, `retrieval_scores`), 과제 01과 같은 `commerce_corpus`, `commerce_train`, `commerce_eval`, `pair_ds`
- **작성:** `# TODO` 아래 `...`를 채우고 문항 끝의 `[자가채점]` 셀을 실행합니다. 1번부터 차례로 풉니다.

In [ ]:
from pathlib import Path

import pandas as pd
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다 (지난 단원과 같은 코드)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 조문이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 이 단원의 임베딩 모델과 e5 계열 프리픽스(질의는 "query: ", 문서는 "passage: ")
MODEL_ID = "intfloat/multilingual-e5-small"
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "

In [ ]:
import json


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def encode_texts(model, texts, prefix):
    """텍스트 목록에 프리픽스를 붙여 정규화된 임베딩(numpy 배열)으로 만듭니다."""
    return model.encode(
        [prefix + text for text in texts],   # 프리픽스를 붙인 문장 목록
        normalize_embeddings=True,           # 길이 1로 정규화해 벡터끼리의 내적이 곧 코사인 유사도가 되게 합니다
        show_progress_bar=False,             # 진행 막대를 끕니다(평가 때마다 여러 번 불러 출력이 길어지지 않게)
    )

In [ ]:
import numpy as np


def gold_ranks(model, corpus, evalset, query_prefix=QUERY_PREFIX, passage_prefix=PASSAGE_PREFIX):
    """평가 질의마다 정답 문서가 유사도 순으로 몇 위인지(1위부터) 담은 배열을 돌려줍니다."""
    doc_ids = [doc["doc_id"] for doc in corpus]
    doc_emb = encode_texts(model, [doc["text"] for doc in corpus], passage_prefix)
    # 학습·평가 파일은 같은 세 열을 쓰며, 검색 평가는 질의와 정답 ID를 사용합니다.
    query_emb = encode_texts(model, [row["anchor"] for row in evalset], query_prefix)

    # 정규화한 벡터끼리의 내적이 코사인 유사도입니다. sims는 (질의 수, 문서 수) 행렬입니다
    sims = query_emb @ doc_emb.T

    # 질의마다 유사도가 높은 순으로 문서 id를 늘어놓고, 정답 문서가 몇 번째인지 셉니다
    ranks = []
    for i, row in enumerate(evalset):
        ranked_ids = [doc_ids[j] for j in np.argsort(-sims[i])]
        ranks.append(ranked_ids.index(row["doc_id"]) + 1)
    return np.array(ranks)


def retrieval_scores(ranks):
    """정답 순위 배열로 Recall@1, Recall@5, MRR을 계산합니다."""
    return {
        "Recall@1": round(float(np.mean(ranks <= 1)), 3),  # 정답 문서가 1위인 질의의 비율
        "Recall@5": round(float(np.mean(ranks <= 5)), 3),  # 정답 문서가 5위 안에 든 질의의 비율
        "MRR": round(float(np.mean(1 / ranks)), 3),        # 평균(1 / 정답 순위)
    }

In [ ]:
from datasets import Dataset

# 과제 01 1번과 같은 조문, 학습 쌍, 평가 질의
commerce_corpus = load_jsonl(data_dir / "과제" / "commerce_corpus.jsonl")
commerce_train = load_jsonl(data_dir / "과제" / "commerce_train.jsonl")
commerce_eval = load_jsonl(data_dir / "과제" / "commerce_eval.jsonl")

# 과제 01 3번과 같은 학습 데이터셋(anchor 에 질의 프리픽스, positive 에 문서 프리픽스)
pair_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in commerce_train],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in commerce_train],
})
display(pair_ds)

## 1. 베이스 모델과 쌍 학습 모델 불러오기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> hard negative를 뽑을 모델과 비교 기준을 준비하기 위해, 베이스 모델과 과제 01에서 저장한 쌍 학습 모델을 불러옵니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) <code style="font-size:0.95em;color:#345473;">MODEL_ID</code>로 만들어 <code style="font-size:0.95em;color:#345473;">device</code>에 올린 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">base_model</code></strong> 을 만듭니다.</li>
<li>(2) <code style="font-size:0.95em;color:#345473;">str(output_dir / &quot;commerce_tuned&quot;)</code> 폴더에서 불러와 <code style="font-size:0.95em;color:#345473;">device</code>에 올린 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">commerce_tuned_model</code></strong> 을 만듭니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01 6-1 저장하고 다시 불러오기 셀과 같은 방법입니다. 모델 이름 대신 저장 폴더 경로를 넘깁니다.
```

</details>

In [ ]:
from sentence_transformers import SentenceTransformer

# 모델을 처음 내려받을 때 HF_TOKEN 경고가 나올 수 있습니다. 로그인 없이 공개 모델을 받는다는 안내라 그대로 진행합니다

# TODO (1): MODEL_ID 로 베이스 모델을 만들어 device 에 올리세요
base_model = ...

# TODO (2): 과제 01에서 저장한 output_dir / "commerce_tuned" 폴더에서 쌍 학습 모델을 불러오세요
commerce_tuned_model = ...

In [ ]:
# [자가채점] 저장 폴더와 불러온 두 모델을 확인합니다
assert (output_dir / "commerce_tuned" / "config_sentence_transformers.json").exists(), "output/commerce_tuned 에 저장한 모델이 없습니다. 과제 01을 7번까지 실행하세요."
assert isinstance(base_model, SentenceTransformer) and isinstance(commerce_tuned_model, SentenceTransformer), "두 모델 모두 SentenceTransformer 입니다. `...`를 채웠는지 확인하세요."
assert not torch.equal(next(base_model.parameters()), next(commerce_tuned_model.parameters())), "commerce_tuned_model의 가중치가 베이스 모델과 같습니다. MODEL_ID 가 아니라 output_dir / \"commerce_tuned\" 폴더에서 불러오세요."
print("✅ 통과!")

## 2. 한 질의에서 hard negative 직접 뽑기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 정답 조문과 헷갈릴 만큼 비슷한 오답 조문(hard negative)이 무엇인지 보기 위해, 학습 질의 하나에서 정답 조문을 뺀 1위 조문을 베이스 모델로 직접 고릅니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) <code style="font-size:0.95em;color:#345473;">base_model</code>로 조문 문장(<code style="font-size:0.95em;color:#345473;">commerce_corpus</code>의 <code style="font-size:0.95em;color:#345473;">text</code>, <code style="font-size:0.95em;color:#345473;">PASSAGE_PREFIX</code>)을 인코딩해 조문 벡터 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">base_doc_emb</code></strong> 를 만듭니다.</li>
<li>(2) 셀에 적어 둔 <code style="font-size:0.95em;color:#345473;">query_idx</code>의 질의(<code style="font-size:0.95em;color:#345473;">commerce_train[26]</code>: &quot;반품했더니 위약금 2만 원을 내라고 해요.&quot;, 정답 조문 <code style="font-size:0.95em;color:#345473;">전자상거래법 제18조 제9항</code>) 하나를 <code style="font-size:0.95em;color:#345473;">QUERY_PREFIX</code>로 인코딩하고(질의 문장은 <code style="font-size:0.95em;color:#345473;">&quot;anchor&quot;</code> 키), (1)의 <code style="font-size:0.95em;color:#345473;">base_doc_emb</code>와의 코사인 유사도 행렬 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">query_sims</code></strong>(numpy 배열, 모양 (1, 39))를 만듭니다.</li>
<li>(3) 유사도가 높은 조문부터 보며 doc_id가 정답 조문과 다른 첫 조문의 문장(<code style="font-size:0.95em;color:#345473;">commerce_corpus</code>의 <code style="font-size:0.95em;color:#345473;">text</code>)을 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">hard_neg_text</code></strong> 에 담습니다.</li>
<li>(4) 그 조문의 doc_id(셀에 적어 둔 <code style="font-size:0.95em;color:#345473;">doc_ids</code>에서 꺼냄)를 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">hard_neg_id</code></strong> 에 담습니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02 3-1 학습 질의와 조문의 유사도 셀과 같은 방법입니다.
```

</details>

In [ ]:
# 조문 id 목록(그대로 씁니다)
doc_ids = [doc["doc_id"] for doc in commerce_corpus]

# TODO (1): 베이스 모델로 조문 벡터 (39, 384) 를 만드세요
base_doc_emb = ...

# 반품 위약금을 묻는 학습 질의(그대로 씁니다)
query_idx = 26

# TODO (2): 이 질의 하나와 조문의 유사도 (1, 39) 를 계산하세요
query_sims = ...

# 유사도가 높은 조문부터 봅니다(그대로 씁니다)
for j in np.argsort(-query_sims[0]):
    # TODO (3): 정답 조문이 아닌 첫 조문의 문장을 hard_neg_text 에 담으세요
    # TODO (4): 그 조문의 doc_id 를 hard_neg_id 에 담고 반복을 멈추세요
    ...

print("정답 조문:", commerce_train[query_idx]["doc_id"], "| hard negative:", hard_neg_id)

In [ ]:
# [자가채점] 조문 벡터와 유사도 행렬의 모양, 고른 조문이 정답 조문을 뺀 1위인지 확인합니다
import hashlib

assert isinstance(base_doc_emb, np.ndarray) and base_doc_emb.shape[0] == 39, f"base_doc_emb의 모양이 {np.shape(base_doc_emb)}입니다. 조문 39개의 벡터 (39, 384) 입니다. commerce_corpus 의 text 39개를 인코딩했는지 확인하세요."
assert isinstance(query_sims, np.ndarray) and query_sims.shape == (1, 39), f"query_sims의 모양이 {np.shape(query_sims)}입니다. 질의 하나와 조문 39개의 유사도 (1, 39) 입니다. 질의를 리스트 하나에 담아 인코딩했는지 확인하세요."
assert hard_neg_id in doc_ids and hard_neg_id != commerce_train[query_idx]["doc_id"], f"hard_neg_id가 {hard_neg_id}입니다. 정답 조문이 아닌 조문의 doc_id 를 담으세요."
assert hard_neg_text == commerce_corpus[doc_ids.index(hard_neg_id)]["text"], "hard_neg_text가 hard_neg_id 조문의 문장과 다릅니다. 같은 위치 j 에서 꺼내세요(doc_id 가 아니라 문장)."
hard_score = query_sims[0, doc_ids.index(hard_neg_id)]
for doc_id, score in zip(doc_ids, query_sims[0]):
    if doc_id != commerce_train[query_idx]["doc_id"]:
        assert score <= hard_score, "정답 조문이 아닌데 hard negative 보다 유사도가 높은 조문이 있습니다. 유사도가 높은 순서로 보며 첫 조문에서 멈췄는지 확인하세요."
# 베이스 모델로 이 질의를 계산하면 정해진 조문과 유사도가 나옵니다(조문 이름은 해시 지문으로 대조)
assert hashlib.sha256(hard_neg_id.encode("utf-8")).hexdigest()[:12] == "d039ba5a094d", f"hard_neg_id가 {hard_neg_id}입니다. base_model 로, commerce_train[query_idx] 질의 하나를 인코딩했는지 확인하세요."
assert abs(hard_score - 0.8575) < 0.002, f"hard negative 의 유사도가 {hard_score:.4f}입니다. 질의에는 QUERY_PREFIX, 조문에는 PASSAGE_PREFIX 를 붙여 base_model 로 인코딩했는지 확인하세요."
print("✅ 통과!")

## 3. mine_hard_negatives로 거르며 뽑기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 점수 규칙으로 후보를 선별해 학습 질의에 오답 후보를 붙이는 triplet 데이터셋을 만듭니다. 마이닝에 쓰는 모델에 따라 남는 행이 달라지므로 두 모델로 한 번씩 뽑습니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) 조문 문장 앞에 <code style="font-size:0.95em;color:#345473;">PASSAGE_PREFIX</code>를 붙인 리스트 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">corpus_texts</code></strong>(조문 순서 그대로, 39개)를 만듭니다.</li>
<li>(2) <code style="font-size:0.95em;color:#345473;">mine_hard_negatives</code>로 <code style="font-size:0.95em;color:#345473;">pair_ds</code>의 질의마다 <code style="font-size:0.95em;color:#345473;">corpus_texts</code>에서 오답 문서를 뽑아 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">base_mined</code></strong>(모델은 <code style="font-size:0.95em;color:#345473;">base_model</code>)를 만듭니다. 정답 점수가 양수일 때 그 95%보다 높은 후보는 버리고(교안 02 4-1의 <code style="font-size:0.95em;color:#345473;">relative_margin</code> 규칙), 질의마다 오답 문서 하나를, 남은 후보 가운데 점수가 가장 높은 것으로 뽑도록 인자를 정하고, <code style="font-size:0.95em;color:#345473;">batch_size=64</code>, <code style="font-size:0.95em;color:#345473;">verbose=False</code>도 넘깁니다. 프리픽스를 붙여 주는 인자는 넘기지 않습니다.</li>
<li>(3) 같은 호출에서 모델만 <code style="font-size:0.95em;color:#345473;">commerce_tuned_model</code>로 바꿔 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">triplet_ds</code></strong>(열 <code style="font-size:0.95em;color:#345473;">anchor</code>, <code style="font-size:0.95em;color:#345473;">positive</code>, <code style="font-size:0.95em;color:#345473;">negative</code>)를 만듭니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02 4-1 점수 간격으로 후보 선별과 9절 선택 실험 셀과 같은 방법입니다. 두 호출은 모델 인자만 다릅니다.
```

</details>

In [ ]:
from sentence_transformers.util import mine_hard_negatives

# TODO (1): 조문 39개에 문서 프리픽스를 붙인 오답 문서 후보 목록을 만드세요
corpus_texts = ...

# TODO (2): 베이스 모델로 오답 문서를 뽑으세요(정답 점수의 95%보다 높은 후보는 버립니다)
base_mined = ...
print("베이스 모델로 오답 문서를 받은 행:", len(base_mined), "/", len(pair_ds))

In [ ]:
# TODO (3): 같은 호출에서 모델만 쌍 학습 모델로 바꿔 triplet 데이터셋을 만드세요
triplet_ds = ...
print("쌍 학습 모델로 오답 문서를 받은 행:", len(triplet_ds), "/", len(pair_ds))

In [ ]:
# [자가채점] 후보 목록, 두 마이닝 결과의 행 수, triplet 데이터셋의 열과 행을 확인합니다
assert isinstance(corpus_texts, list) and len(corpus_texts) == 39, "corpus_texts는 조문 39개의 문장 리스트입니다. commerce_corpus 의 조문마다 문장 하나씩 담았는지 확인하세요."
for text in corpus_texts:
    assert text.startswith(PASSAGE_PREFIX) and not text.startswith(PASSAGE_PREFIX + PASSAGE_PREFIX), "corpus_texts의 문장마다 passage: 를 한 번 붙이세요."
assert isinstance(base_mined, Dataset) and isinstance(triplet_ds, Dataset), "base_mined, triplet_ds는 mine_hard_negatives 가 돌려준 Dataset 입니다. `...`를 mine_hard_negatives 호출로 채웠는지 확인하세요."
assert len(base_mined) <= len(pair_ds) and 0 < len(triplet_ds) <= len(pair_ds), "질의마다 오답 하나를 뽑으므로 행 수는 pair_ds 이하이며 학습할 triplet이 있어야 합니다."
assert triplet_ds.column_names == ["anchor", "positive", "negative"], f"triplet_ds의 열이 {triplet_ds.column_names}입니다. 질의마다 오답 문서 하나를 뽑도록 했는지 확인하세요."
for anchor, positive, negative in zip(triplet_ds["anchor"], triplet_ds["positive"], triplet_ds["negative"]):
    assert anchor.startswith(QUERY_PREFIX) and not anchor.startswith(QUERY_PREFIX + QUERY_PREFIX), "anchor는 pair_ds 와 같이 query: 로 한 번만 시작합니다. 학습 데이터셋으로 pair_ds 를 넘겼는지 확인하세요."
    assert negative in corpus_texts, "negative가 corpus_texts 의 문장이 아닙니다. 오답 후보로 corpus_texts 를 넘겼는지 확인하세요."
    assert negative != positive, "negative가 positive 와 같은 행이 있습니다. 정답 문서를 후보에 남기는 인자를 넣지 않았는지 확인하세요."

# 쌍 학습 모델의 점수로 행마다 오답 문서 점수가 정답 문서 점수의 95% 이하인지 봅니다(거짓 오답 거르기)
# anchor, positive, negative 에는 프리픽스가 이미 붙어 있어 프리픽스 자리에 빈 문자열을 넘깁니다
anchor_emb = encode_texts(commerce_tuned_model, triplet_ds["anchor"], "")
positive_emb = encode_texts(commerce_tuned_model, triplet_ds["positive"], "")
negative_emb = encode_texts(commerce_tuned_model, triplet_ds["negative"], "")
positive_scores = (anchor_emb * positive_emb).sum(axis=1)
negative_scores = (anchor_emb * negative_emb).sum(axis=1)
for positive_score, negative_score in zip(positive_scores, negative_scores):
    assert negative_score <= positive_score * 0.95 + 1e-4, f"오답 문서 점수 {negative_score:.3f}가 정답 문서 점수 {positive_score:.3f}의 95%보다 높은 행이 있습니다. 정답 점수의 95%보다 높은 후보를 버리는 인자를 넣었는지 확인하세요."
print("✅ 통과!")

## 4. 검토 목록의 거짓 오답 행 빼기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 점수 규칙(정답 점수의 95%)이 거르지 못한 거짓 오답 행을 학습 전에 빼려고, 조문을 읽고 확인한 짝 목록으로 triplet 데이터셋을 거릅니다.</p></div>

- **같은 법 안의 거짓 오답:** 쇼핑 조문은 모두 전자상거래법이라 교안 02의 다른 법 인용 규칙으로는 찾을 수 없고, 같은 법 안에서 거짓 오답이 생깁니다.
- **품절 환불:** 제15조 제4항(품절 환불)은 "제18조제1항부터 제5항까지의 규정을 준용"하므로, 품절 환불의 지연배상금을 묻는 질의에는 제18조 제2항(지연배상금)도 맞습니다.
- **판매자 연락처:** 오픈마켓에서 판매자 연락처를 묻는 질의에는 제20조 제2항(중개자가 판매자 신원정보 제공)과 제13조 제1항(판매자가 광고에 전화번호 등 표시)이 함께 맞습니다.
- **검토 목록의 범위:** 이번 검토에서는 다음 순서의 짝을 가진 행을 학습에서 보류합니다. 예를 들어 쇼핑몰 광고에 적을 판매자 정보를 묻는 질의(정답 제13조 제1항)에는 중개자의 의무인 제20조 제2항이 답이 아니고, 청약철회 환불 지연을 묻는 질의(정답 제18조 제2항)에는 품절 환불 조문인 제15조 제4항이 답이 아닙니다. 검토 목록 밖의 행까지 거짓 오답이 없다고 보장하지는 않습니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) 조문 <code style="font-size:0.95em;color:#345473;">text</code> 앞에 <code style="font-size:0.95em;color:#345473;">PASSAGE_PREFIX</code>를 붙인 문장을 키로, <code style="font-size:0.95em;color:#345473;">commerce_corpus</code>의 조문 딕셔너리를 값으로 하는 딕셔너리 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">passage_to_doc</code></strong> 을 만듭니다.</li>
<li>(2) <code style="font-size:0.95em;color:#345473;">triplet_ds</code>의 행마다 (정답 조문 doc_id, 오답 조문 doc_id)가 셀에 적어 둔 <code style="font-size:0.95em;color:#345473;">same_answer_pairs</code>에 있으면 키가 <code style="font-size:0.95em;color:#345473;">&quot;질의&quot;</code>(<code style="font-size:0.95em;color:#345473;">QUERY_PREFIX</code>를 뗀 질의), <code style="font-size:0.95em;color:#345473;">&quot;정답 조문&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;오답 조문&quot;</code>(이 순서)인 딕셔너리를 리스트 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">removed_rows</code></strong> 에 더하고, 없으면 <code style="font-size:0.95em;color:#345473;">triplet_ds</code>의 행(row)을 그대로 리스트 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">kept_rows</code></strong> 에 더해 두 리스트를 만듭니다. 모든 입력 행을 두 리스트 가운데 하나에 한 번씩 담습니다.</li>
<li>(3) <code style="font-size:0.95em;color:#345473;">kept_rows</code>로 다시 만든 Dataset(열은 <code style="font-size:0.95em;color:#345473;">anchor</code>, <code style="font-size:0.95em;color:#345473;">positive</code>, <code style="font-size:0.95em;color:#345473;">negative</code> 그대로)을 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">triplet_ds</code></strong> 에 다시 담습니다.</li>
<li>(4) <code style="font-size:0.95em;color:#345473;">removed_rows</code>로 만든 DataFrame <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">removed_table</code></strong> 을 만듭니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02 4-3 셀에서 검토 목록 부분(in_list)만 쓰는 방법입니다.
```

</details>

In [ ]:
# 이번 검토에서는 이 순서의 짝을 가진 행을 학습에서 보류합니다.
same_answer_pairs = [
    ("전자상거래법 제15조 제4항", "전자상거래법 제18조 제2항"),
    ("전자상거래법 제20조 제2항", "전자상거래법 제13조 제1항"),
]

# TODO (1): 프리픽스를 붙인 조문 text 로 조문 딕셔너리를 찾는 passage_to_doc 을 만드세요
passage_to_doc = ...

mined_row_count = len(triplet_ds)  # [제공] 모든 입력 행을 남긴 행과 보류한 행으로 나눕니다.
kept_rows = []
removed_rows = []
for row in triplet_ds:
    # TODO (2): 정답 조문과 오답 조문의 doc_id 짝이 same_answer_pairs 에 있으면 removed_rows 에, 없으면 kept_rows 에 더하세요
    ...

# TODO (3): 남은 행 kept_rows 로 triplet 데이터셋을 다시 만드세요
triplet_ds = ...
print("뺀 행:", len(removed_rows), "| 학습에 쓸 행:", len(triplet_ds))

In [ ]:
# 뺀 행 수는 쌍 학습 모델에 따라 달라 빈 테이블일 수도 있습니다
# TODO (4): 뺀 행 removed_rows 로 테이블을 만드세요
removed_table = ...
display(removed_table)

In [ ]:
# [자가채점] 조문 찾기 딕셔너리, 뺀 행의 키와 짝, 남은 triplet 데이터셋을 확인합니다
assert isinstance(passage_to_doc, dict), "passage_to_doc은 딕셔너리입니다. `...`를 채웠는지 확인하세요."
assert len(passage_to_doc) == 39, f"passage_to_doc의 키가 {len(passage_to_doc)}개입니다. commerce_corpus 의 조문 39개마다 키 하나씩 담았는지 확인하세요."
for passage, doc in passage_to_doc.items():
    assert passage.startswith(PASSAGE_PREFIX) and not passage.startswith(PASSAGE_PREFIX + PASSAGE_PREFIX), "passage_to_doc의 키는 조문 text 앞에 passage: 를 한 번 붙인 문장입니다. 키를 만드는 식을 확인하세요."
    assert isinstance(doc, dict) and "doc_id" in doc, "passage_to_doc의 값은 commerce_corpus 의 조문 딕셔너리입니다. 값으로 doc_id 가 아니라 조문 딕셔너리를 담았는지 확인하세요."
assert isinstance(removed_rows, list) and isinstance(kept_rows, list), "removed_rows, kept_rows는 리스트입니다. 셀에 적어 둔 빈 리스트에 append 로 더했는지 확인하세요."
for removed in removed_rows:
    assert isinstance(removed, dict), "removed_rows의 행은 딕셔너리입니다. 질의, 정답 조문, 오답 조문을 키로 담은 딕셔너리를 더하세요."
    assert list(removed) == ["질의", "정답 조문", "오답 조문"], f"removed_rows 행의 키가 {list(removed)}입니다. 질의, 정답 조문, 오답 조문 순서로 넣었는지 확인하세요."
    assert (removed["정답 조문"], removed["오답 조문"]) in same_answer_pairs, f"뺀 행의 짝 ({removed['정답 조문']}, {removed['오답 조문']})이 same_answer_pairs 에 없습니다. 짝이 목록에 있는 행만 removed_rows 에 더하세요."
    assert not removed["질의"].startswith(QUERY_PREFIX), "removed_rows의 질의는 앞의 query: 를 뗀 문장입니다. anchor 에서 QUERY_PREFIX 길이만큼 떼었는지 확인하세요."
assert isinstance(triplet_ds, Dataset), "triplet_ds는 Dataset 입니다. kept_rows 로 Dataset 을 다시 만들어 담았는지 확인하세요."
assert triplet_ds.column_names == ["anchor", "positive", "negative"], f"triplet_ds의 열이 {triplet_ds.column_names}입니다. kept_rows 에 triplet_ds 의 행을 그대로 더했는지 확인하세요."
assert len(kept_rows) + len(removed_rows) == mined_row_count, "검토 전의 모든 행을 kept_rows와 removed_rows로 빠짐없이 나누세요."
assert len(triplet_ds) == len(kept_rows), f"triplet_ds가 {len(triplet_ds)}행, kept_rows가 {len(kept_rows)}행입니다. kept_rows 로 다시 만든 Dataset 을 triplet_ds 에 담았는지 확인하세요."
for positive, negative in zip(triplet_ds["positive"], triplet_ds["negative"]):
    pair = (passage_to_doc[positive]["doc_id"], passage_to_doc[negative]["doc_id"])
    assert pair not in same_answer_pairs, f"triplet_ds에 검토 목록의 짝 {pair} 행이 남아 있습니다. 짝이 목록에 있는 행은 kept_rows 가 아니라 removed_rows 에 더하세요."
assert isinstance(removed_table, pd.DataFrame) and len(removed_table) == len(removed_rows), "removed_table은 removed_rows 를 pd.DataFrame 에 넘겨 만든 테이블입니다. 행 수가 removed_rows 와 같은지 확인하세요."
print("✅ 통과!")

## 5. triplet으로 hard negative 학습

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> hard negative를 추가했을 때의 검색 변화를 비교하기 위해, 쌍 학습 모델을 이어 학습하지 않고 <code style="font-size:0.95em;color:#345473;">MODEL_ID</code>에서 새로 시작해 4번에서 거른 <code style="font-size:0.95em;color:#345473;">triplet_ds</code>로 학습합니다. 데이터·배치 구성과 조기 종료 시점이 달라 실제 갱신 횟수가 같다고 가정하지 않습니다.</p></div>

최대 5에폭 동안 매 에폭 검증·저장하고, 검증 `Recall@5`가 2회 연속 개선되지 않으면 일찍 종료합니다. 학습이 끝나면 **검증 Recall@5가 가장 높았던 체크포인트**를 복원합니다. 검증용 `commerce_eval`은 모델 선택에 사용하므로 독립적인 최종 테스트 점수로 해석하지 않으며, 일반 83질의는 모델 선택에 쓰지 않습니다.

검증 쌍의 MNR 손실 `eval_loss`도 로그에 기록합니다. 이 손실은 검증 배치 구성의 영향을 받으므로, 손실 최저가 아니라 전체 조문 검색의 Recall@5로 모델을 선택합니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) <code style="font-size:0.95em;color:#345473;">MODEL_ID</code>로 새로 만들어 <code style="font-size:0.95em;color:#345473;">device</code>에 올린 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">commerce_hard_model</code></strong>(<code style="font-size:0.95em;color:#345473;">base_model</code>, <code style="font-size:0.95em;color:#345473;">commerce_tuned_model</code>과 다른 객체)을 만듭니다.</li>
<li>(2) <code style="font-size:0.95em;color:#345473;">commerce_hard_model</code>을 넘긴 <code style="font-size:0.95em;color:#345473;">MultipleNegativesRankingLoss</code>로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">loss</code></strong> 를 만듭니다.</li>
<li>(3) 아래 값의 <code style="font-size:0.95em;color:#345473;">SentenceTransformerTrainingArguments</code>를 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">args</code></strong> 에 담습니다.</li>
</ul></div>

| 인자 | 값 |
|---|---|
| `output_dir` | `str(output_dir / "commerce_hard_run")` |
| `num_train_epochs` | 5(최대) |
| `per_device_train_batch_size` | 16 |
| `per_device_eval_batch_size` | 32 |
| `learning_rate` | 5e-5 |
| `warmup_steps` | 0.1 |
| `batch_sampler` | `BatchSamplers.NO_DUPLICATES` |
| `eval_strategy` | `"epoch"` |
| `save_strategy` | `"epoch"` |
| `load_best_model_at_end` | `True` |
| `metric_for_best_model` | `"eval_commerce_cosine_recall@5"` |
| `greater_is_better` | `True` |
| `save_total_limit` | 2 |
| `logging_steps` | 5 |
| `report_to` | `"none"` |
| `seed` | 42 |
| `use_cpu` | `device.type == "cpu"` |

- (4) `model=commerce_hard_model`, `args=args`, `train_dataset=triplet_ds`, `eval_dataset=eval_ds`, `loss=loss`, `evaluator=retrieval_evaluator`, `callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]`를 넘긴 `SentenceTransformerTrainer`를 **`trainer`** 에 담습니다. `eval_ds`와 `retrieval_evaluator`는 제공 코드입니다.
- (5) `trainer`로 학습한 반환값을 **`train_output`** 에 담습니다. 제공 코드의 `epoch_history`는 에폭별 `eval_loss`·검증 Recall@5를, `best_checkpoint_table`은 종료 에폭·최적 에폭·체크포인트·최적 검증 Recall@5를 표시합니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 5-2~5-3의 모델·손실·검증·학습 설정과 같은 순서입니다.
eval_dataset은 검증 손실, retrieval_evaluator는 전체 조문 검색 점수를 계산합니다. 콜백은 검증 Recall@5의 개선을 기다리는 횟수를 정합니다.
```

</details>

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import cos_sim
from transformers import EarlyStoppingCallback

# [제공] 검증 쌍은 eval_loss를 기록하는 데만 쓰며 가중치 갱신에는 쓰지 않습니다.
eval_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in commerce_eval],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in commerce_eval],
})
# 전체 쇼핑 조문에서 검증 질의의 정답 순위를 구해 Recall@5로 모델을 선택합니다.
retrieval_evaluator = InformationRetrievalEvaluator(
    # 질의 ID는 정답 매핑과 연결할 문자열 키이고, 값은 프리픽스가 붙은 질문입니다.
    queries={str(i): QUERY_PREFIX + row["anchor"] for i, row in enumerate(commerce_eval)},
    # 문서 ID -> 프리픽스가 붙은 검색 문서입니다.
    corpus={doc["doc_id"]: PASSAGE_PREFIX + doc["text"] for doc in commerce_corpus},
    # 질의 ID -> 정답 조문 ID 집합입니다.
    relevant_docs={str(i): {row["doc_id"]} for i, row in enumerate(commerce_eval)},
    name="commerce",  # 지표 이름에 포함되어 eval_commerce_cosine_recall@5를 만듭니다.
    score_functions={"cosine": cos_sim},  # 문서 점수는 코사인 유사도로 비교합니다.
    precision_recall_at_k=[1, 5],  # Recall@1과 Recall@5를 계산합니다.
    batch_size=64,  # 평가 문장을 64개씩 인코딩합니다.
    write_csv=False,
)

# TODO (1): MODEL_ID 에서 새 hard negative 학습 모델을 만드세요
commerce_hard_model = ...

# TODO (2): 새 모델의 MNR 손실을 만드세요
loss = ...

# TODO (3): 검증·저장·최적 모델 복원에 필요한 빈칸을 요구사항 테이블대로 채우세요
args = SentenceTransformerTrainingArguments(
    output_dir=str(output_dir / "commerce_hard_run"),
    num_train_epochs=...,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=...,
    learning_rate=5e-5,
    warmup_steps=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy=...,
    save_strategy=...,
    load_best_model_at_end=...,
    # evaluator가 만든 검증 지표 이름입니다. name·점수 방식·Recall@K 설정에서 정해집니다.
    metric_for_best_model=...,
    greater_is_better=...,                     # Recall@5는 값이 클수록 좋은 모델입니다.
    save_total_limit=...,
    logging_steps=5,
    report_to="none",
    seed=42,
    use_cpu=(device.type == "cpu"),  # GPU(cuda)이면 False, CPU이면 True
)

# TODO (4): 검증 데이터, 검색 평가기, patience 2 콜백을 채워 학습기를 만드세요
trainer = SentenceTransformerTrainer(
    model=commerce_hard_model,
    args=args,
    train_dataset=triplet_ds,
    eval_dataset=...,
    loss=loss,
    evaluator=...,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=...)],
)

In [ ]:
# 학습을 시작하면 [transformers] 토크나이저 PAD/BOS/EOS 안내가 한 번 나옵니다. 모델 설정을 토크나이저에 맞췄다는 안내라 오류가 아닙니다.
# TODO (5): 학습하세요. 반환값의 global_step 은 실제 갱신 횟수입니다.
train_output = ...
display(train_output)

# [제공] 에폭별 검증 손실과 검색 점수, 복원한 최적 체크포인트를 확인합니다.
eval_logs = [log for log in trainer.state.log_history if "eval_commerce_cosine_recall@5" in log]
epoch_history = pd.DataFrame([
    {"에폭": log["epoch"], "검증 MNR 손실": log["eval_loss"],
     "검증 Recall@5": log["eval_commerce_cosine_recall@5"]}
    for log in eval_logs
])
display(epoch_history.round(3))
best_step = int(Path(trainer.state.best_model_checkpoint).name.rsplit("-", 1)[1])
best_epoch = next(log["epoch"] for log in eval_logs if log["step"] == best_step)
best_checkpoint_table = pd.DataFrame([{
    "선택 기준": "검증 Recall@5 최대",
    "최적 Recall@5": trainer.state.best_metric,
    "복원 체크포인트": trainer.state.best_model_checkpoint,
    "종료 에폭": trainer.state.epoch,
    "최적 에폭": best_epoch,
}])
display(best_checkpoint_table.round(3))

In [ ]:
# [자가채점] 설정과 실제 검증 기록을 확인합니다. 조기 종료와 점수의 상승·하락을 허용합니다.
assert isinstance(commerce_hard_model, SentenceTransformer) and commerce_hard_model is not base_model, "commerce_hard_model은 MODEL_ID로 새로 만든 모델입니다. 베이스 모델을 학습하지 마세요."
assert commerce_hard_model is not commerce_tuned_model, "commerce_hard_model은 쌍 학습 모델을 이어 학습하지 않고 새로 만듭니다."
assert isinstance(loss, MultipleNegativesRankingLoss) and loss.model is commerce_hard_model, "loss에 현재 학습할 모델을 넘겼는지 확인하세요."
assert isinstance(args, SentenceTransformerTrainingArguments), "args는 SentenceTransformerTrainingArguments입니다."
assert args.batch_sampler == BatchSamplers.NO_DUPLICATES, "batch_sampler=BatchSamplers.NO_DUPLICATES를 넣으세요."
assert args.num_train_epochs == 5 and args.per_device_train_batch_size == 16 and args.per_device_eval_batch_size == 32, "최대 5에폭, 학습 배치 16, 검증 배치 32를 넣으세요."
assert abs(args.learning_rate - 5e-5) < 1e-12 and args.warmup_steps == 0.1, "learning_rate=5e-5, warmup_steps=0.1을 넣으세요."
assert args.eval_strategy == "epoch" and args.save_strategy == "epoch", "매 에폭 검증·저장하도록 eval_strategy와 save_strategy를 모두 epoch으로 둡니다."
assert args.load_best_model_at_end and args.metric_for_best_model == "eval_commerce_cosine_recall@5" and args.greater_is_better, "검증 Recall@5가 가장 높은 체크포인트를 학습 종료 후 복원하도록 설정하세요."
assert args.save_total_limit == 2 and args.logging_steps == 5 and args.seed == 42, "save_total_limit=2, logging_steps=5, seed=42를 넣으세요."
assert len(args.report_to) == 0 and args.use_cpu == (device.type == "cpu"), "report_to=none, use_cpu=(device.type == cpu)의 요구사항을 확인하세요."
assert args.output_dir.endswith("commerce_hard_run"), "output_dir에 요구사항의 실행 폴더 경로를 넣으세요."
assert isinstance(trainer, SentenceTransformerTrainer) and trainer.model is commerce_hard_model and trainer.args is args, "trainer에 현재 모델과 args를 넘기세요."
assert trainer.train_dataset.column_names == ['anchor', 'positive', 'negative'] and len(trainer.train_dataset) == len(triplet_ds), "train_dataset에 triplet_ds를 넘기세요. 검증 쌍은 학습 데이터가 아닙니다."
assert trainer.eval_dataset.column_names == ["anchor", "positive"] and len(trainer.eval_dataset) == 46, "eval_dataset에 제공한 두 열짜리 eval_ds 46행을 넘기세요."
assert trainer.evaluator is retrieval_evaluator, "전체 commerce_corpus를 검색하는 제공 retrieval_evaluator를 trainer에 넘기세요."
assert any(isinstance(callback, EarlyStoppingCallback) and callback.early_stopping_patience == 2 for callback in trainer.callback_handler.callbacks), "callbacks에 patience 2인 EarlyStoppingCallback을 넣으세요."
assert 0 < train_output.global_step <= 30 and 0 < trainer.state.epoch <= 5, "학습이 실행됐는지 확인하세요. 최대 5에폭 안에서 조기 종료할 수 있습니다."
assert 1 <= len(epoch_history) <= 5 and epoch_history["검증 MNR 손실"].notna().all(), "매 에폭 검증 손실이 기록되도록 eval_dataset과 eval_strategy를 확인하세요."
assert trainer.state.best_model_checkpoint and Path(trainer.state.best_model_checkpoint).is_dir(), "최적 체크포인트가 저장되지 않았습니다. epoch 저장·검증과 load_best_model_at_end를 확인하세요."
assert abs(trainer.state.best_metric - epoch_history["검증 Recall@5"].max()) < 1e-12, "최적 검증 점수는 기록된 Recall@5의 최댓값이어야 합니다. eval_loss로 선택하지 마세요."
print("✅ 통과!")

## 6. 세 모델 비교

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> hard negative 학습이 어느 질의 묶음의 점수를 바꾸는지 보기 위해, 세 모델의 점수를 전체, 학습에 나온 조문 질의, 처음 보는 조문 질의로 나눠 한 테이블에 모읍니다. 여기서 처음 보는 조문은 원래 쌍 학습의 정답 기준이며, Triplet 학습에서는 오답 후보로 접했을 수 있습니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) 과제 01 6번과 같은 doc_id 집합 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">train_doc_ids</code></strong> 를 만듭니다.</li>
<li>(2) 과제 01 6번과 같은 불리언 배열 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">is_unseen</code></strong>(<code style="font-size:0.95em;color:#345473;">commerce_eval</code> 순서, 길이 46)을 만듭니다.</li>
<li>(3) <code style="font-size:0.95em;color:#345473;">gold_ranks</code>(프리픽스 인자는 기본값)로 세 모델의 정답 순위 배열 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">base_ranks</code></strong>, <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">tuned_ranks</code></strong>, <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">hard_ranks</code></strong> 를 만듭니다.</li>
<li>(4) 셀에 적어 둔 <code style="font-size:0.95em;color:#345473;">query_groups</code>, <code style="font-size:0.95em;color:#345473;">model_ranks</code> 순서로 묶음마다 모델별 한 행씩, 열이 <code style="font-size:0.95em;color:#345473;">&quot;질의 묶음&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;모델&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;질의 수&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;Recall@1&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;Recall@5&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;MRR&quot;</code>(이 순서)인 아홉 행 테이블 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">compare_table</code></strong>(점수는 그 묶음의 순위만 골라 <code style="font-size:0.95em;color:#345473;">retrieval_scores</code>로 계산)을 만듭니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02 6-1~6-2 질의 묶음별 세 모델 비교 셀과 같은 방법입니다.
과제 01 6번 테이블에 묶음 "전체"와 모델 하나를 더한 모양이고, 전체 묶음은 모두 True 인 불리언 배열로 고릅니다.
```

</details>

In [ ]:
# TODO (1): 학습 쌍의 doc_id 집합을 만드세요
train_doc_ids = ...

# TODO (2): 평가 질의마다 정답 조문이 train_doc_ids 에 없으면 True 인 배열을 만드세요
is_unseen = ...

# TODO (3): 같은 조문과 평가 질의로 세 모델의 정답 순위를 구하세요
base_ranks = ...
tuned_ranks = ...
hard_ranks = ...

In [ ]:
# 질의 묶음과 모델(그대로 씁니다). 전체 묶음은 모두 True 인 배열입니다
query_groups = {
    "전체": np.full(len(commerce_eval), True),
    "학습에 나온 조문 질의": ~is_unseen,
    "처음 보는 조문 질의": is_unseen,
}
model_ranks = {"베이스 모델": base_ranks, "쌍 학습 모델": tuned_ranks, "hard negative 학습 모델": hard_ranks}

# TODO (4): 묶음과 모델마다 점수를 계산해 행 딕셔너리를 compare_rows 에 더하세요
compare_rows = []
for group, mask in query_groups.items():
    for name, ranks in model_ranks.items():
        ...
compare_table = pd.DataFrame(compare_rows)
display(compare_table)

In [ ]:
# [자가채점] 세 순위 배열, 처음 보는 조문 질의 표시, 비교 테이블의 행과 열과 값을 확인합니다
model_names = ["베이스 모델", "쌍 학습 모델", "hard negative 학습 모델"]
group_names = ["전체", "학습에 나온 조문 질의", "처음 보는 조문 질의"]
metrics = ["Recall@1", "Recall@5", "MRR"]
for ranks in [base_ranks, tuned_ranks, hard_ranks]:
    assert isinstance(ranks, np.ndarray) and ranks.shape == (46,), f"순위 배열의 모양이 {np.shape(ranks)}입니다. gold_ranks 의 반환값을 담으세요."
    assert ranks.min() >= 1 and ranks.max() <= 39, "순위는 1부터 조문 수 39까지입니다. gold_ranks 의 반환값을 그대로 담았는지 확인하세요."
assert isinstance(train_doc_ids, set) and len(train_doc_ids) == 27, "train_doc_ids는 commerce_train 의 doc_id 27개를 모은 집합(set)입니다. 평가 질의가 아니라 학습 쌍의 doc_id 로 만들었는지 확인하세요."
assert isinstance(is_unseen, np.ndarray) and is_unseen.dtype == bool and is_unseen.shape == (46,), "is_unseen은 평가 질의 46개마다 True/False 인 numpy 배열입니다. 리스트를 np.array 로 감쌌는지 확인하세요."
assert list(np.flatnonzero(is_unseen)) == [2, 8, 10, 11, 21, 22, 25, 26, 27], "is_unseen의 True 위치가 다릅니다. doc_id가 train_doc_ids 에 없을 때 True 이고, commerce_eval 순서 그대로입니다."
assert isinstance(compare_table, pd.DataFrame), "compare_table은 pd.DataFrame 입니다. 행 딕셔너리 리스트 compare_rows 를 pd.DataFrame 에 넘겼는지 확인하세요."
assert list(compare_table.columns) == ["질의 묶음", "모델"] + ["질의 수"] + metrics, f"compare_table의 열이 {list(compare_table.columns)}입니다. 행 딕셔너리의 키를 질의 묶음, 모델, 질의 수, Recall@1, Recall@5, MRR 순서로 넣었는지 확인하세요."
assert list(compare_table["질의 묶음"]) == ["전체"] * 3 + ["학습에 나온 조문 질의"] * 3 + ["처음 보는 조문 질의"] * 3, "질의 묶음 열은 전체, 학습에 나온 조문 질의, 처음 보는 조문 질의 순서로 세 행씩입니다. 바깥 for 문이 query_groups 인지 확인하세요."
assert list(compare_table["모델"]) == model_names * 3, "모델 열은 묶음마다 베이스 모델, 쌍 학습 모델, hard negative 학습 모델 순서입니다. 안쪽 for 문이 model_ranks 인지 확인하세요."
assert list(compare_table["질의 수"]) == [46] * 3 + [37] * 3 + [9] * 3, f"질의 수가 {list(compare_table['질의 수'])}입니다. 묶음의 True 개수를 넣으세요."
assert compare_table[metrics].to_numpy().min() >= 0 and compare_table[metrics].to_numpy().max() <= 1, "점수는 0~1 입니다. retrieval_scores 가 돌려준 값을 그대로 넣었는지 확인하세요."
for _, row in compare_table.iterrows():
    expected_scores = retrieval_scores(model_ranks[row["모델"]][query_groups[row["질의 묶음"]]])
    assert row[metrics].to_dict() == expected_scores, "각 행은 해당 모델·질의 묶음의 순위만 골라 계산해야 합니다. 묶음 점수가 같아지는 것은 허용합니다."
# 같은 점수가 나오는 것도 허용하고, 모델별 실제 순위와 일치하는지 확인합니다.
for name, model in [("베이스 모델", base_model), ("쌍 학습 모델", commerce_tuned_model), ("hard negative 학습 모델", commerce_hard_model)]:
    assert np.array_equal(model_ranks[name], gold_ranks(model, commerce_corpus, commerce_eval)), f"{name}의 순위가 이 모델의 실제 검색 결과와 다릅니다."
print("✅ 통과!")

## 7. 일반 분야 검색의 퇴행 비교

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 온라인쇼핑에 맞춘 학습이 다른 분야의 검색 점수를 얼마나 바꾸었는지 확인하기 위해, 베이스 모델·쌍 학습 모델·hard negative 학습 모델을 학습에 쓰지 않은 일반 질문 83개로 비교합니다. 점수는 내려가거나 그대로이거나 올라갈 수 있습니다.</p></div>

일반 검색 문서는 총 433개입니다. 관광 43질의는 관광 388문서, 상식 30질의는 상식 30문서, 가상 도서관 10질의는 도서관 15문서 안에서만 검색합니다. 이 작은 검색 묶음의 점수를 모델의 일반능력 전체로 해석하지 않습니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) 제공한 반복문에서 현재 모델 <code style="font-size:0.95em;color:#345473;">model</code>, 현재 분야의 문서 <code style="font-size:0.95em;color:#345473;">domain_corpus</code>, 질의 <code style="font-size:0.95em;color:#345473;">domain_eval</code>을 <code style="font-size:0.95em;color:#345473;">gold_ranks</code>에 넘깁니다(프리픽스는 기본값). 1부터 시작하는 정수 순위를 <code style="font-size:0.95em;color:#345473;">ranks[domain_mask]</code>에 담아, 모델마다 별도의 길이 83인 배열을 <code style="font-size:0.95em;color:#345473;">general_model_ranks</code>에 저장합니다. 분야를 합쳐 검색하거나 질의 순서를 바꾸지 않습니다.</li>
</ul></div>

준비와 집계 코드는 제공합니다. `general_score_table`은 전체·분야별 `Recall@1`, `Recall@5`, `MRR`과 베이스 대비 `Recall@5 변화(pp)`를 보여 줍니다. 변화폭은 반올림 전 원순위로 계산하며, 음수는 하락·0은 동일·양수는 상승입니다. 자가채점은 모델별 순위를 다시 계산해 대조하므로 시간이 조금 걸립니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 6-3 일반 분야 검색 셀과 같습니다.
반복문이 모델과 분야별 후보를 준비해 두므로, 해당 모델의 정답 순위를 구하는 공용 함수 호출을 채웁니다.
```

</details>

In [ ]:
# [제공] 과제만 새 커널에서 실행해도 같은 일반 평가 파일을 읽습니다.
general_corpus = load_jsonl(data_dir / "본실습" / "general_corpus.jsonl")
general_eval = load_jsonl(data_dir / "본실습" / "general_eval.jsonl")
# eval 행에는 분야 열이 없으므로 정답 doc_id로 문서의 분야를 연결합니다.
general_doc_domain = {doc["doc_id"]: doc["domain"] for doc in general_corpus}
general_domains = np.array([general_doc_domain[row["doc_id"]] for row in general_eval])
general_domain_names = ["관광", "상식", "도서관"]
general_doc_counts = {domain: sum(doc["domain"] == domain for doc in general_corpus) for domain in general_domain_names}
general_data_table = pd.DataFrame([
    {"분야": domain, "검색 문서 수": general_doc_counts[domain], "평가 질의 수": int(np.sum(general_domains == domain))}
    for domain in general_domain_names
])
display(general_data_table)

In [ ]:
# 모델과 분야를 순서대로 평가합니다. 모델마다 새 순위 배열을 만듭니다.
general_models = [("베이스 모델", base_model), ("쌍 학습 모델", commerce_tuned_model), ("hard negative 학습 모델", commerce_hard_model)]
general_model_ranks = {}
for name, model in general_models:
    ranks = np.empty(len(general_eval), dtype=int)
    for domain in general_domain_names:
        domain_mask = general_domains == domain
        domain_corpus = [doc for doc in general_corpus if doc["domain"] == domain]
        domain_eval = [row for row, keep in zip(general_eval, domain_mask) if keep]
        # TODO (1): 현재 모델로 같은 분야 문서에서 질의별 정답 순위를 구하세요
        ranks[domain_mask] = ...
    general_model_ranks[name] = ranks

In [ ]:
# [제공] 교안과 같은 집계입니다. 전체는 분야별 검색 순위 83개를 합쳐 계산합니다.
base_general_ranks = general_model_ranks["베이스 모델"]
general_groups = {"전체(일반 분야)": np.full(len(general_eval), True)}
general_groups.update({domain: general_domains == domain for domain in general_domain_names})
general_rows = []
for group, mask in general_groups.items():
    before_r5 = float(np.mean(base_general_ranks[mask] <= 5))
    for name, ranks in general_model_ranks.items():
        scores = retrieval_scores(ranks[mask])
        general_rows.append({
            "평가 묶음": group, "모델": name,
            "문서 수": len(general_corpus) if group == "전체(일반 분야)" else general_doc_counts[group],
            "질의 수": int(mask.sum()), **scores,
            # 표시용으로 반올림한 Recall@5끼리 빼지 않고, 원래 순위에서 변화폭을 구합니다.
            "Recall@5 변화(pp)": round(100 * (float(np.mean(ranks[mask] <= 5)) - before_r5), 2),
        })
general_score_table = pd.DataFrame(general_rows)
display(general_score_table)

In [ ]:
# [자가채점] 순위의 형태·후보 범위와 집계의 일관성을 확인합니다. 점수의 향상·하락은 채점하지 않습니다.
expected_general_models = dict([("베이스 모델", base_model), ("쌍 학습 모델", commerce_tuned_model), ("hard negative 학습 모델", commerce_hard_model)])
assert list(general_model_ranks) == list(expected_general_models), "general_model_ranks에 제공한 모델 이름과 순서를 유지하세요."
assert len(general_corpus) == 433 and len(general_eval) == 83, "data/본실습의 general_corpus.jsonl과 general_eval.jsonl을 읽었는지 확인하세요."
for name, ranks in general_model_ranks.items():
    assert isinstance(ranks, np.ndarray) and ranks.shape == (83,) and np.issubdtype(ranks.dtype, np.integer), f"{name}: gold_ranks의 정수 순위 83개를 원래 질의 순서에 담으세요."
    for domain in general_domain_names:
        part = ranks[general_domains == domain]
        assert part.min() >= 1 and part.max() <= general_doc_counts[domain], f"{name} {domain}: 현재 분야의 문서만 domain_corpus로 넘기고 1부터 시작하는 순위를 담으세요."
for i, name in enumerate(general_model_ranks):
    for other in list(general_model_ranks)[:i]:
        assert not np.shares_memory(general_model_ranks[name], general_model_ranks[other]), "모델마다 새 순위 배열을 만들어야 합니다. 한 배열을 여러 모델에 재사용하지 마세요."
assert isinstance(general_score_table, pd.DataFrame) and list(general_score_table.columns) == ["평가 묶음", "모델", "문서 수", "질의 수", "Recall@1", "Recall@5", "MRR", "Recall@5 변화(pp)"], "제공한 집계 셀의 general_score_table 열과 순서를 유지하세요."
assert list(zip(general_score_table["평가 묶음"], general_score_table["모델"])) == [(group, name) for group in general_groups for name in expected_general_models], "전체, 관광, 상식, 도서관 순서로 모델마다 한 행이 있어야 합니다."
for _, row in general_score_table.iterrows():
    mask = general_groups[row["평가 묶음"]]
    assert row["질의 수"] == int(mask.sum()), "행마다 해당 묶음의 질의 수를 넣어야 합니다."
    expected_docs = 433 if row["평가 묶음"] == "전체(일반 분야)" else general_doc_counts[row["평가 묶음"]]
    assert row["문서 수"] == expected_docs, "전체는 문서 합계 433개, 분야 행은 해당 분야의 후보 수입니다."
    assert all(0 <= row[metric] <= 1 for metric in ["Recall@1", "Recall@5", "MRR"]), "세 검색 지표는 0~1 범위여야 합니다."
    after_hits = np.count_nonzero(general_model_ranks[row["모델"]][mask] <= 5)
    before_hits = np.count_nonzero(general_model_ranks["베이스 모델"][mask] <= 5)
    assert abs(row["Recall@5 변화(pp)"] - round(100 * (after_hits - before_hits) / int(mask.sum()), 2)) < 1e-9, "Recall@5 변화(pp)는 반올림한 점수의 차이가 아니라 원순위의 정답 수 차이로 계산합니다."
for name in expected_general_models:
    model_rows = general_score_table[general_score_table["모델"] == name]
    whole = model_rows[model_rows["평가 묶음"] == "전체(일반 분야)"].iloc[0]
    parts = model_rows[model_rows["평가 묶음"] != "전체(일반 분야)"]
    for metric in ["Recall@1", "Recall@5", "MRR"]:
        combined = (parts[metric] * parts["질의 수"]).sum() / parts["질의 수"].sum()
        assert abs(combined - whole[metric]) < 0.002, f"{name}: 전체 {metric}는 분야별 질의 수로 가중한 점수와 맞아야 합니다."
# 모델별 실제 검색 결과와 대조합니다. 점수 방향이나 고정 점수는 요구하지 않습니다.
for name, model in expected_general_models.items():
    for domain in general_domain_names:
        domain_mask = general_domains == domain
        domain_corpus = [doc for doc in general_corpus if doc["domain"] == domain]
        domain_eval = [row for row, keep in zip(general_eval, domain_mask) if keep]
        expected_ranks = gold_ranks(model, domain_corpus, domain_eval)
        assert np.array_equal(general_model_ranks[name][domain_mask], expected_ranks), f"{name} {domain}: 이 모델의 실제 검색 순위와 다릅니다. 현재 model과 분야별 문서·질의를 사용했는지 확인하세요."
print("✅ 통과!")